# Predicting Bank Customer Churn for Retention Strategies
### BDM Capstone Project — Final Submission — Code Notebook

**Author:** Agni Prasanth M (24ds1000065@ds.study.iitm.ac.in)
**Dataset:** [Customer Churn Analysis Dataset — Kaggle](https://www.kaggle.com/datasets/gregorykipngeno/customer-churn-analysis-dataset)

This notebook reproduces every result, table, and figure presented in the final report:
1. Setup & Data Loading
2. Data Cleaning & Quality Checks
3. Exploratory Data Analysis (Descriptive Statistics & Segment-Level Churn Rates)
4. Feature Engineering
5. Preprocessing, Train/Test Split & SMOTE
6. Model Training — Full-Feature Model (incl. `Complain`)
7. Model Training — Behavioural Model (excl. `Complain`) — Primary Model
8. Model Evaluation (ROC, Precision-Recall, Confusion Matrix)
9. SHAP Interpretability
10. Business Insight Charts (EDA visualisations used in the report)
11. ROI Simulation


## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # switch to 'inline' / remove this line if running interactively in Jupyter
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_curve, precision_recall_curve, auc)
from imblearn.over_sampling import SMOTE
import shap

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11})
np.random.seed(42)


In [2]:
# Load the dataset (update the path if running locally)
df = pd.read_excel('customer_churn_csv.xlsx')
print("Shape:", df.shape)
df.head()


Shape: (10000, 18)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,Credit Card,IsActiveMember,EstimatedSalary,Churned,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


## 2. Data Cleaning & Quality Checks

Standard checks for missing values, duplicate records, and type consistency, as documented in
Section 4.1 of the report.

In [3]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate CustomerId:", df['CustomerId'].duplicated().sum())
print()
print("Data types:")
print(df.dtypes)


Missing values per column:
RowNumber             0
CustomerId            0
Surname               0
CreditScore           0
Geography             0
Gender                0
Age                   0
Tenure                0
Balance               0
NumOfProducts         0
Credit Card           0
IsActiveMember        0
EstimatedSalary       0
Churned               0
Complain              0
Satisfaction Score    0
Card Type             0
Point Earned          0
dtype: int64

Full-row duplicates: 0
Duplicate CustomerId: 0

Data types:
RowNumber               int64
CustomerId              int64
Surname                   str
CreditScore             int64
Geography                 str
Gender                    str
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
Credit Card             int64
IsActiveMember          int64
EstimatedSalary       float64
Churned                 int64
Complain                int64
Satisfaction Scor

In [4]:
# Outlier / range screening on CreditScore and Balance
print(df[['CreditScore', 'Balance']].describe())
print()
zero_balance_count = (df['Balance'] == 0).sum()
print(f"Zero-balance accounts: {zero_balance_count} ({zero_balance_count/len(df)*100:.2f}% of customers)")


        CreditScore        Balance
count  10000.000000   10000.000000
mean     650.528800   76485.889288
std       96.653299   62397.405202
min      350.000000       0.000000
25%      584.000000       0.000000
50%      652.000000   97198.540000
75%      718.000000  127644.240000
max      850.000000  250898.090000

Zero-balance accounts: 3617 (36.17% of customers)


## 3. Exploratory Data Analysis — Descriptive Statistics

In [5]:
# Overall churn rate
print("Churn distribution:")
print(df['Churned'].value_counts())
print(df['Churned'].value_counts(normalize=True).round(4) * 100)


Churn distribution:
Churned
0    7962
1    2038
Name: count, dtype: int64
Churned
0    79.62
1    20.38
Name: proportion, dtype: float64


In [6]:
# Descriptive statistics for numeric variables (Table 2 in report)
numeric_cols = ['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary',
                 'Satisfaction Score','Point Earned']
desc = df[numeric_cols].describe().T
desc['median'] = df[numeric_cols].median()
desc['range'] = desc['max'] - desc['min']
desc[['mean','median','std','min','max','range']].round(2)


,mean,median,std,min,max,range
CreditScore,650.53,652.00,96.65,350.00,850.00,500.00
Age,38.92,37.00,10.49,18.00,92.00,74.00
Tenure,5.01,5.00,2.89,0.00,10.00,10.00
Balance,76485.89,97198.54,62397.41,0.00,250898.09,250898.09
NumOfProducts,1.53,1.00,0.58,1.00,4.00,3.00
EstimatedSalary,100090.24,100193.92,57510.49,11.58,199992.48,199980.90
Satisfaction Score,3.01,3.00,1.41,1.00,5.00,4.00
Point Earned,606.52,605.00,225.92,119.00,1000.00,881.00


In [7]:
# Categorical distributions (Table 3 in report)
for col in ['Geography', 'Gender', 'Card Type']:
    print(f"\n{col} distribution:")
    counts = df[col].value_counts()
    pct = df[col].value_counts(normalize=True).round(4) * 100
    print(pd.concat([counts, pct], axis=1, keys=['Count', '%']))



Geography distribution:
           Count      %
Geography              
France      5014  50.14
Germany     2509  25.09
Spain       2477  24.77

Gender distribution:
        Count      %
Gender              
Male     5457  54.57
Female   4543  45.43

Card Type distribution:
           Count      %
Card Type              
DIAMOND     2507  25.07
GOLD        2502  25.02
SILVER      2496  24.96
PLATINUM    2495  24.95


In [8]:
# Churn rate by key segments (Section 5 of report)
segment_cols = ['Geography', 'Gender', 'Card Type', 'IsActiveMember',
                 'NumOfProducts', 'Complain', 'Satisfaction Score']
for col in segment_cols:
    print(f"\n--- Churn rate by {col} ---")
    print((df.groupby(col)['Churned'].mean() * 100).round(2))



--- Churn rate by Geography ---
Geography
France     16.17
Germany    32.44
Spain      16.67
Name: Churned, dtype: float64

--- Churn rate by Gender ---
Gender
Female    25.07
Male      16.47
Name: Churned, dtype: float64

--- Churn rate by Card Type ---
Card Type
DIAMOND     21.78
GOLD        19.26
PLATINUM    20.36
SILVER      20.11
Name: Churned, dtype: float64

--- Churn rate by IsActiveMember ---
IsActiveMember
0    26.87
1    14.27
Name: Churned, dtype: float64

--- Churn rate by NumOfProducts ---
NumOfProducts
1     27.71
2      7.60
3     82.71
4    100.00
Name: Churned, dtype: float64

--- Churn rate by Complain ---
Complain
0     0.05
1    99.51
Name: Churned, dtype: float64

--- Churn rate by Satisfaction Score ---
Satisfaction Score
1    20.03
2    21.80
3    19.64
4    20.62
5    19.81
Name: Churned, dtype: float64


In [9]:
# Age-group churn rates (Figure 4)
df['AgeGroup'] = pd.cut(df['Age'], bins=[17,30,40,50,60,100],
                         labels=['18-30','31-40','41-50','51-60','60+'])
print(df.groupby('AgeGroup', observed=True)['Churned'].agg(['mean','count']))


              mean  count
AgeGroup                 
18-30     0.075203   1968
31-40     0.121096   4451
41-50     0.339655   2320
51-60     0.562108    797
60+       0.247845    464


In [10]:
# Correlation of numeric/binary features with Churned (Section 4.3, Figure 10 heatmap)
corr_cols = numeric_cols + ['Churned','Complain','IsActiveMember','Credit Card']
corr_with_churn = df[corr_cols].corr()['Churned'].sort_values(ascending=False)
print(corr_with_churn)


Churned               1.000000
Complain              0.995693
Age                   0.285296
Balance               0.118577
EstimatedSalary       0.012490
Point Earned         -0.004628
Satisfaction Score   -0.005849
Credit Card          -0.006976
Tenure               -0.013656
CreditScore          -0.026771
NumOfProducts        -0.047611
IsActiveMember       -0.156356
Name: Churned, dtype: float64


## 4. Feature Engineering

Four engineered features, as documented in Section 4.2 of the report.

In [11]:
df['BalanceSalaryRatio'] = df['Balance'] / df['EstimatedSalary'].replace(0, np.nan)
df['BalanceSalaryRatio'] = df['BalanceSalaryRatio'].fillna(0)

df['ProductActivityFlag'] = ((df['NumOfProducts'] >= 3) & (df['IsActiveMember'] == 0)).astype(int)
df['ZeroBalance'] = (df['Balance'] == 0).astype(int)

# AgeGroup already created above for EDA

print("Engineered features preview:")
df[['BalanceSalaryRatio','ProductActivityFlag','ZeroBalance','AgeGroup']].head()


Engineered features preview:


,BalanceSalaryRatio,ProductActivityFlag,ZeroBalance,AgeGroup
0,0.000000,0,1,41-50
1,0.744677,0,0,41-50
2,1.401375,1,0,41-50
3,0.000000,0,1,31-40
4,1.587055,0,0,41-50


## 5. Preprocessing, Train/Test Split & SMOTE

An 80/20 stratified split preserves the 20.38% churn rate in both partitions. SMOTE is applied
to the **training set only** to correct class imbalance (Section 4.4).

Two feature sets are built:
- **Full-Feature set** — includes `Complain`
- **Behavioural set** — excludes `Complain` (the primary, forward-looking model; see Section 4.3)


In [12]:
feature_cols_num = ['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary',
                     'Satisfaction Score','Point Earned','BalanceSalaryRatio']
feature_cols_bin_full  = ['Credit Card','IsActiveMember','Complain','ProductActivityFlag','ZeroBalance']
feature_cols_bin_behav = ['Credit Card','IsActiveMember','ProductActivityFlag','ZeroBalance']
feature_cols_cat = ['Geography','Gender','Card Type']

y = df['Churned']


In [13]:
def build_and_split(bin_cols):
    X = df[feature_cols_num + bin_cols + feature_cols_cat]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), feature_cols_num),
        ('bin', 'passthrough', bin_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), feature_cols_cat)
    ])

    X_train_proc = preprocessor.fit_transform(X_train)
    X_test_proc  = preprocessor.transform(X_test)

    feature_names = (feature_cols_num + bin_cols +
                      list(preprocessor.named_transformers_['cat'].get_feature_names_out(feature_cols_cat)))

    sm = SMOTE(random_state=42)
    X_train_sm, y_train_sm = sm.fit_resample(X_train_proc, y_train)

    return dict(X_train_proc=X_train_proc, X_test_proc=X_test_proc,
                y_train=y_train, y_test=y_test,
                X_train_sm=X_train_sm, y_train_sm=y_train_sm,
                feature_names=feature_names, preprocessor=preprocessor)

data_full  = build_and_split(feature_cols_bin_full)
data_behav = build_and_split(feature_cols_bin_behav)

print("Before SMOTE:", np.bincount(data_behav['y_train']))
print("After SMOTE :", np.bincount(data_behav['y_train_sm']))


Before SMOTE: [6370 1630]
After SMOTE : [6370 6370]


## 6. Model Training

Three algorithms — Logistic Regression (baseline), Random Forest, and XGBoost — are trained
and tuned via `GridSearchCV` with stratified k-fold cross-validation, optimising AUC-ROC
(Section 4.5). The routine below is run once for the **Full-Feature** set and once for the
**Behavioural** set.

In [14]:
def train_evaluate(data, label):
    X_train_sm, y_train_sm = data['X_train_sm'], data['y_train_sm']
    X_test_proc, y_test = data['X_test_proc'], data['y_test']

    results = {}
    fitted = {}

    # --- Logistic Regression (baseline) ---
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train_sm, y_train_sm)
    lr_proba = lr.predict_proba(X_test_proc)[:, 1]
    lr_pred  = lr.predict(X_test_proc)
    results['LogisticRegression'] = dict(
        auc=roc_auc_score(y_test, lr_proba), precision=precision_score(y_test, lr_pred),
        recall=recall_score(y_test, lr_pred), f1=f1_score(y_test, lr_pred),
        cm=confusion_matrix(y_test, lr_pred).tolist())
    fitted['lr'] = lr
    fitted['lr_proba'] = lr_proba

    # --- Random Forest (GridSearchCV) ---
    rf_grid = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {'n_estimators': [150, 250], 'max_depth': [10, None]},
        cv=StratifiedKFold(3), scoring='roc_auc', n_jobs=-1
    )
    rf_grid.fit(X_train_sm, y_train_sm)
    rf_best = rf_grid.best_estimator_
    rf_proba = rf_best.predict_proba(X_test_proc)[:, 1]
    rf_pred  = rf_best.predict(X_test_proc)
    results['RandomForest'] = dict(
        best_params=rf_grid.best_params_, auc=roc_auc_score(y_test, rf_proba),
        precision=precision_score(y_test, rf_pred), recall=recall_score(y_test, rf_pred),
        f1=f1_score(y_test, rf_pred), cm=confusion_matrix(y_test, rf_pred).tolist())
    fitted['rf'] = rf_best
    fitted['rf_proba'] = rf_proba

    # --- XGBoost (GridSearchCV) ---
    xgb_grid = GridSearchCV(
        XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
        {'n_estimators': [150, 250], 'max_depth': [4, 6], 'learning_rate': [0.1]},
        cv=StratifiedKFold(3), scoring='roc_auc', n_jobs=-1
    )
    xgb_grid.fit(X_train_sm, y_train_sm)
    xgb_best = xgb_grid.best_estimator_
    xgb_proba = xgb_best.predict_proba(X_test_proc)[:, 1]
    xgb_pred  = xgb_best.predict(X_test_proc)
    results['XGBoost'] = dict(
        best_params=xgb_grid.best_params_, auc=roc_auc_score(y_test, xgb_proba),
        precision=precision_score(y_test, xgb_pred), recall=recall_score(y_test, xgb_pred),
        f1=f1_score(y_test, xgb_pred), cm=confusion_matrix(y_test, xgb_pred).tolist())
    fitted['xgb'] = xgb_best
    fitted['xgb_proba'] = xgb_proba

    print(f"===== {label} =====")
    for k, v in results.items():
        print(k, {kk: vv for kk, vv in v.items() if kk != 'cm'})

    return results, fitted


In [15]:
results_full, fitted_full = train_evaluate(data_full, "FULL-FEATURE MODEL (incl. Complain)")


===== FULL-FEATURE MODEL (incl. Complain) =====
LogisticRegression {'auc': 0.9988468691496699, 'precision': 0.9975429975429976, 'recall': 0.9950980392156863, 'f1': 0.996319018404908}
RandomForest {'best_params': {'max_depth': None, 'n_estimators': 250}, 'auc': 0.9989115306926791, 'precision': 0.9975429975429976, 'recall': 0.9950980392156863, 'f1': 0.996319018404908}
XGBoost {'best_params': {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 250}, 'auc': 0.996822347029264, 'precision': 0.9975429975429976, 'recall': 0.9950980392156863, 'f1': 0.996319018404908}


In [16]:
results_behav, fitted_behav = train_evaluate(data_behav, "BEHAVIOURAL MODEL (excl. Complain) — PRIMARY MODEL")


===== BEHAVIOURAL MODEL (excl. Complain) — PRIMARY MODEL =====
LogisticRegression {'auc': 0.7933047590895655, 'precision': 0.42251461988304095, 'recall': 0.7083333333333334, 'f1': 0.5293040293040293}
RandomForest {'best_params': {'max_depth': None, 'n_estimators': 250}, 'auc': 0.862770500788255, 'precision': 0.6832844574780058, 'recall': 0.571078431372549, 'f1': 0.6221628838451269}
XGBoost {'best_params': {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 250}, 'auc': 0.860406505567051, 'precision': 0.6827794561933535, 'recall': 0.553921568627451, 'f1': 0.6116373477672531}


## 7. Model Performance Summary (Tables 4 & 5 in the report)

In [17]:
def results_table(results):
    return pd.DataFrame({
        model: {'AUC-ROC': v['auc'], 'Precision': v['precision'],
                'Recall': v['recall'], 'F1-Score': v['f1']}
        for model, v in results.items()
    }).T.round(4)

print("Full-Feature Model (incl. Complain):")
display(results_table(results_full))

print("\nBehavioural Model (excl. Complain) — Primary Model:")
display(results_table(results_behav))


Full-Feature Model (incl. Complain):


,AUC-ROC,Precision,Recall,F1-Score
LogisticRegression,0.9988,0.9975,0.9951,0.9963
RandomForest,0.9989,0.9975,0.9951,0.9963
XGBoost,0.9968,0.9975,0.9951,0.9963



Behavioural Model (excl. Complain) — Primary Model:


,AUC-ROC,Precision,Recall,F1-Score
LogisticRegression,0.7933,0.4225,0.7083,0.5293
RandomForest,0.8628,0.6833,0.5711,0.6222
XGBoost,0.8604,0.6828,0.5539,0.6116


## 8. Model Evaluation Plots — ROC, Precision-Recall, Confusion Matrix

In [18]:
y_test = data_behav['y_test']

plt.figure(figsize=(7,6))
for name, proba, color in [('Logistic Regression', fitted_behav['lr_proba'], '#4C72B0'),
                             ('Random Forest',       fitted_behav['rf_proba'], '#55A868'),
                             ('XGBoost',              fitted_behav['xgb_proba'], '#C44E52')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})", color=color, linewidth=2)
plt.plot([0,1],[0,1],'k--', alpha=0.4, label='Random Classifier')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Behavioural Churn Model')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [19]:
plt.figure(figsize=(7,6))
for name, proba, color in [('Logistic Regression', fitted_behav['lr_proba'], '#4C72B0'),
                             ('Random Forest',       fitted_behav['rf_proba'], '#55A868'),
                             ('XGBoost',              fitted_behav['xgb_proba'], '#C44E52')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    plt.plot(rec, prec, label=name, color=color, linewidth=2)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves — Behavioural Churn Model')
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()


In [20]:
rf_pred = (fitted_behav['rf_proba'] >= 0.5).astype(int)
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
plt.title('Confusion Matrix — Random Forest (Behavioural Model)')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## 9. SHAP Interpretability

`TreeExplainer` on the tuned XGBoost Behavioural Model produces exact Shapley values,
ranking churn drivers and showing the direction of each feature's impact (Section 4.7,
Section 5.7 of the report).

In [21]:
X_test_df = pd.DataFrame(data_behav['X_test_proc'], columns=data_behav['feature_names'])

explainer = shap.TreeExplainer(fitted_behav['xgb'])
shap_values = explainer.shap_values(X_test_df)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_ranking = pd.Series(mean_abs_shap, index=data_behav['feature_names']).sort_values(ascending=False)
print("SHAP feature importance ranking:")
print(shap_ranking)


SHAP feature importance ranking:
Age                    1.017976
NumOfProducts          0.855522
IsActiveMember         0.455905
Gender_Male            0.302792
Geography_Germany      0.278649
Balance                0.273479
Satisfaction Score     0.188469
EstimatedSalary        0.188070
BalanceSalaryRatio     0.181943
Tenure                 0.181564
CreditScore            0.170204
Point Earned           0.133546
Credit Card            0.103827
Card Type_GOLD         0.095099
Geography_Spain        0.079413
Card Type_SILVER       0.069068
Card Type_PLATINUM     0.057634
ZeroBalance            0.034308
ProductActivityFlag    0.000635
dtype: float32


In [22]:
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=True)


In [23]:
shap.summary_plot(shap_values, X_test_df, show=True)


## 10. Business Insight Charts (EDA visualisations used in the report)

In [24]:
# Figure 1 & 2: Overall churn distribution + churn by geography
fig, axes = plt.subplots(1, 2, figsize=(13,5))

counts = df['Churned'].value_counts().sort_index()
axes[0].bar(['Retained (0)','Churned (1)'], counts.values, color=['#4C72B0','#C44E52'])
for i, v in enumerate(counts.values):
    axes[0].text(i, v+50, f"{v}\n({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')
axes[0].set_title('Overall Churn Distribution')

geo = df.groupby('Geography')['Churned'].mean().sort_values(ascending=False) * 100
bars = axes[1].bar(geo.index, geo.values, color=sns.color_palette('viridis', len(geo)))
for b, v in zip(bars, geo.values):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.5, f"{v:.1f}%", ha='center', fontweight='bold')
axes[1].set_title('Churn Rate by Geography'); axes[1].set_ylabel('Churn Rate (%)')
plt.tight_layout()
plt.show()


In [25]:
# Figure 3 & 4: Churn by number of products + age group
fig, axes = plt.subplots(1, 2, figsize=(13,5))

prod = df.groupby('NumOfProducts')['Churned'].mean() * 100
bars = axes[0].bar(prod.index.astype(str), prod.values, color=sns.color_palette('rocket', len(prod)))
for b, v in zip(bars, prod.values):
    axes[0].text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}%", ha='center', fontweight='bold')
axes[0].set_title('Churn Rate by Number of Products'); axes[0].set_xlabel('Number of Products')

age = df.groupby('AgeGroup', observed=True)['Churned'].mean() * 100
bars = axes[1].bar(age.index.astype(str), age.values, color=sns.color_palette('mako', len(age)))
for b, v in zip(bars, age.values):
    axes[1].text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}%", ha='center', fontweight='bold')
axes[1].set_title('Churn Rate by Age Group')
plt.tight_layout()
plt.show()


In [26]:
# Figure: Correlation heatmap of numeric features
plt.figure(figsize=(9,7))
numcols = ['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary',
           'Satisfaction Score','Point Earned','Complain','IsActiveMember','Churned']
corr = df[numcols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink':0.8})
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()


In [27]:
# Age distribution & Balance boxplot by churn status
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.histplot(data=df, x='Age', hue='Churned', bins=30, kde=True,
             palette=['#4C72B0','#C44E52'], multiple='layer', alpha=0.5, ax=axes[0])
axes[0].set_title('Age Distribution by Churn Status')

sns.boxplot(data=df, x='Churned', y='Balance', hue='Churned',
            palette=['#4C72B0','#C44E52'], legend=False, ax=axes[1])
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['Retained','Churned'])
axes[1].set_title('Balance Distribution by Churn Status')
plt.tight_layout()
plt.show()


## 11. ROI Simulation (Section 6.4 of the report)

Estimates the financial impact of a targeted 15% reduction in churn, using the documented
assumptions from Section 4.8: annual revenue at 2.5% of balance, a EUR 50 retention-campaign
cost per customer, and an acquisition cost of 6x the retention cost.

In [28]:
n_churned = int(df['Churned'].sum())
avg_balance_churned = df[df['Churned'] == 1]['Balance'].mean()

# Documented assumptions
nim_fee_rate = 0.025                          # 2.5% of balance as annual revenue proxy
retention_campaign_cost_per_customer = 50      # EUR
acquisition_cost_multiplier = 6                # 5-7x per proposal, midpoint used
target_reduction = 0.15

annual_revenue_per_customer = avg_balance_churned * nim_fee_rate
customers_retained = n_churned * target_reduction
revenue_protected = customers_retained * annual_revenue_per_customer

acquisition_cost_per_new_customer = retention_campaign_cost_per_customer * acquisition_cost_multiplier
acquisition_cost_avoided = customers_retained * acquisition_cost_per_new_customer

retention_campaign_total_cost = n_churned * retention_campaign_cost_per_customer
net_benefit = revenue_protected + acquisition_cost_avoided - retention_campaign_total_cost
roi_multiple = (revenue_protected + acquisition_cost_avoided) / retention_campaign_total_cost

print(f"Currently churned customers:            {n_churned:,}")
print(f"Avg. balance of churned customers:       EUR {avg_balance_churned:,.2f}")
print(f"Est. annual revenue per customer (2.5%): EUR {annual_revenue_per_customer:,.2f}")
print(f"Customers retained at 15% reduction:     {customers_retained:,.1f}")
print(f"Annual revenue protected:                EUR {revenue_protected:,.2f}")
print(f"Acquisition cost avoided:                EUR {acquisition_cost_avoided:,.2f}")
print(f"Retention campaign cost (all at-risk):    EUR {retention_campaign_total_cost:,.2f}")
print(f"Net annual benefit:                      EUR {net_benefit:,.2f}")
print(f"Return multiple:                         {roi_multiple:.2f}x")


Currently churned customers:            2,038
Avg. balance of churned customers:       EUR 91,109.48
Est. annual revenue per customer (2.5%): EUR 2,277.74
Customers retained at 15% reduction:     305.7
Annual revenue protected:                EUR 696,304.17
Acquisition cost avoided:                EUR 91,710.00
Retention campaign cost (all at-risk):    EUR 101,900.00
Net annual benefit:                      EUR 686,114.17
Return multiple:                         7.73x


In [29]:
categories = ['Retention\nCampaign Cost', 'Revenue\nProtected', 'Acquisition Cost\nAvoided', 'Net\nBenefit']
values = [-retention_campaign_total_cost, revenue_protected, acquisition_cost_avoided, net_benefit]
colors = ['#C44E52' if v < 0 else '#55A868' for v in values]

plt.figure(figsize=(8,6))
bars = plt.bar(categories, values, color=colors)
for b, v in zip(bars, values):
    plt.text(b.get_x()+b.get_width()/2, v + (15000 if v >= 0 else -25000),
              f'€{v:,.0f}', ha='center', fontweight='bold', fontsize=9)
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('EUR')
plt.title('ROI Simulation — 15% Churn Reduction Impact')
plt.tight_layout()
plt.show()


---
**End of notebook.** All figures, tables, and metrics above correspond directly to the
figures/tables cited in the final report (`BDM_Capstone_Final_Report.docx`).